# 03 — 2026 Tahminleri

En iyi model ile 2026 YKS soru dağılımı tahmini, önem skorları ve trend analizi.

> **Not:** Bu tahminler geçmiş yıl verilerine dayalı istatistiksel çıkarımlardır. Kesin soru çıkma garantisi vermez.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from predict_2026 import generate_predictions
from config import PREDICTIONS_FILE, SUMMARY_FILE

pred_df = generate_predictions()
pred_df.head()

In [ ]:
print('Tahmin özeti:')
print(pred_df[['session','field','predicted_question_count','importance_label','trend_label','confidence_label']]
      .groupby(['session','field','importance_label']).size().unstack(fill_value=0))

In [ ]:
# JSON özeti
with open(SUMMARY_FILE, encoding='utf-8') as f:
    summary = json.load(f)
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
# SAYISAL - AYT konu önemi
sayisal_ayt = pred_df[(pred_df['session']=='AYT') & (pred_df['field']=='SAYISAL')].copy()
sayisal_ayt = sayisal_ayt.sort_values('importance_score', ascending=True)

colors_map = {'Çok yüksek': '#2ecc71', 'Yüksek': '#3498db', 'Orta': '#f39c12', 'Düşük': '#e74c3c'}
bar_colors = sayisal_ayt['importance_label'].map(colors_map).fillna('#95a5a6')

fig, ax = plt.subplots(figsize=(11, max(6, len(sayisal_ayt)*0.28)))
bars = ax.barh(sayisal_ayt['topic'], sayisal_ayt['importance_score'],
               color=bar_colors, edgecolor='white')
ax.set_xlabel('Önem Skoru (0-100)')
ax.set_title('AYT SAYISAL — Konu Önem Skorları (2026 Tahmini)')
plt.tight_layout()
plt.show()

In [ ]:
# TYT konu tahminleri — ders bazlı
tyt = pred_df[pred_df['session']=='TYT'].copy()
tyt_by_subj = tyt.groupby('subject')['predicted_question_count_rounded'].sum().sort_values(ascending=False)

tyt_by_subj.plot(kind='bar', figsize=(10,5), color='steelblue', edgecolor='white')
plt.title('TYT 2026 — Ders Bazlı Tahmini Soru Sayısı')
plt.ylabel('Tahmini Soru Sayısı')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Trend etiketlerine göre konu sayısı
trend_counts = pred_df.groupby(['session','trend_label']).size().unstack(fill_value=0)
trend_counts.plot(kind='bar', figsize=(10,5), edgecolor='white')
plt.title('Trend Dağılımı (TYT vs AYT)')
plt.ylabel('Konu Sayısı')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Tahmin aralığı görselleştirmesi — SAYISAL AYT Matematik
mat = pred_df[(pred_df['session']=='AYT') & (pred_df['field']=='SAYISAL') & (pred_df['subject']=='Matematik')].copy()
mat = mat.sort_values('predicted_question_count', ascending=False)

fig, ax = plt.subplots(figsize=(11, max(5, len(mat)*0.35)))
ax.barh(mat['topic'], mat['predicted_question_count'], color='#3498db', alpha=0.7, label='Tahmin')
ax.errorbar(
    mat['predicted_question_count'], mat['topic'],
    xerr=[mat['predicted_question_count'] - mat['lower_bound'],
          mat['upper_bound'] - mat['predicted_question_count']],
    fmt='none', color='#2c3e50', capsize=4, linewidth=1.5
)
ax.set_xlabel('Tahmini Soru Sayısı')
ax.set_title('AYT Matematik — Konu Bazlı Tahmini Soru Aralığı (2026)')
plt.tight_layout()
plt.show()